# 01 – Data Exploration (Phase 1)
Read-only inspection of the raw Realtor.com USA listing data. Nothing here modifies `data/raw/`. Reports go to `reports/`.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg") if __name__ == "__main__" else None
import matplotlib.pyplot as plt

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
RAW = ROOT / "data" / "raw" / "realtor-data.zip.csv.zip"
REPORTS = ROOT / "reports"; REPORTS.mkdir(exist_ok=True)
df = pd.read_csv(RAW)   # pandas reads the single CSV inside the zip directly
print(df.shape)
df.dtypes

## Schema and missingness

In [ ]:
schema = pd.DataFrame({"dtype": df.dtypes.astype(str), "missing_pct": (df.isna().mean()*100).round(2),
                       "n_unique": df.nunique()})
schema.to_csv(REPORTS / "phase1_schema_missingness.csv")
schema

## Duplicates
No column is a unique listing id, so `(street, zip_code)` is used as a proxy for a property. `street` is an anonymised numeric code in this dataset.

In [ ]:
addr = df.dropna(subset=["street","zip_code"]).assign(
    addr=lambda d: d.street.astype(int).astype(str) + "|" + d.zip_code.astype(int).astype(str))
g = addr.groupby("addr").size()
dups = {"rows": len(df), "full_row_duplicates": int(df.duplicated().sum()),
        "rows_missing_street_or_zip": int(len(df)-len(addr)),
        "distinct_addresses": int(len(g)), "addresses_with_multiple_rows": int((g>1).sum()),
        "rows_in_repeated_addresses": int(g[g>1].sum()), "max_rows_one_address": int(g.max())}
pd.Series(dups).to_csv(REPORTS / "phase1_duplicates.csv", header=["value"])
dups

## Status and price

In [ ]:
status = df.groupby("status").agg(rows=("price","size"), price_median=("price","median"),
        price_p01=("price", lambda s: s.quantile(.01)), price_p99=("price", lambda s: s.quantile(.99)),
        price_max=("price","max"), prev_sold_missing=("prev_sold_date", lambda s: s.isna().mean()))
status.to_csv(REPORTS / "phase1_status_summary.csv")
status

In [ ]:
num = df[["price","bed","bath","acre_lot","house_size"]]
desc = num.describe(percentiles=[.01,.05,.5,.95,.99]).T
desc.to_csv(REPORTS / "phase1_numeric_summary.csv")
implaus = {"price<=0": int((df.price<=0).sum()), "price<1000": int((df.price<1000).sum()),
           "price>=50M": int((df.price>=5e7).sum()), "bed>20": int((df.bed>20).sum()),
           "bath>20": int((df.bath>20).sum()), "house_size<200": int((df.house_size<200).sum()),
           "house_size>50000": int((df.house_size>50000).sum()), "acre_lot==0": int((df.acre_lot==0).sum()),
           "acre_lot>1000": int((df.acre_lot>1000).sum())}
pd.Series(implaus).to_csv(REPORTS / "phase1_implausible_values.csv", header=["rows"])
desc, implaus

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
p = df.price.dropna(); p = p[p > 0]
ax[0].hist(p.clip(upper=p.quantile(.99)), bins=80); ax[0].set_title("Price (clipped at p99)"); ax[0].set_xlabel("USD")
ax[1].hist(np.log10(p), bins=80); ax[1].set_title("log10(price)")
fig.tight_layout(); fig.savefig(REPORTS / "phase1_price_distribution.png", dpi=130)

## Geography

In [ ]:
state_counts = df.state.value_counts(dropna=False)
state_counts.to_csv(REPORTS / "phase1_state_counts.csv", header=["rows"])
cc = df.groupby(["state","city"]).size().rename("rows").reset_index()
cc.sort_values("rows", ascending=False).to_csv(REPORTS / "phase1_city_counts.csv", index=False)
cover = pd.DataFrame([(t, int((cc.rows>=t).sum()), int(cc.rows[cc.rows>=t].sum())) for t in (10,30,100,500)],
                     columns=["min_rows_per_city","n_cities","rows_covered"])
cover.to_csv(REPORTS / "phase1_city_coverage_thresholds.csv", index=False)
cover

In [ ]:
core = ["price","bed","bath","house_size","acre_lot","city","state","zip_code"]
complete = {"all_rows": len(df), "complete_core": int(df[core].dropna().shape[0]),
            "sold_complete_no_acre": int(df[df.status=="sold"][["price","bed","bath","house_size","city","state"]].dropna().shape[0])}
pd.Series(complete).to_csv(REPORTS / "phase1_complete_cases.csv", header=["rows"])
complete